# ETH Monthly Options Analysis

Monthly options expire on the **last Friday of each month at 1:30 PM IST** (08:00 UTC) .

They are the **deepest, most liquid, lowest-theta** options on the chain. Used by institutions for strategic positioning.

**This notebook analyzes:**

1. The next monthly expiry date in IST
2. Contract count and OI structure
3. IV smile at the monthly tenor
4. Greeks profile (delta, gamma, theta, vega)
5. Theta decay over the month
6. Breakeven per strike
7. Liquidity (deepest of all tenors)
8. Best monthly contract for a buyer
9. Monthly vs weekly vs quarterly comparison
10. The monthly buyer's verdict

## Setup

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from datetime import datetime, timezone, timedelta
import time

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (13, 5)

DERIBIT = 'https://www.deribit.com/api/v2/public'
CURRENCY = 'ETH'
IST_OFFSET = timedelta(hours=5, minutes=30)

def to_ist(dt):
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone(IST_OFFSET))

print('ready')

## 1. Fetch chain and identify monthly expiries

Monthly = **last Friday of the month**. We find all Friday expiries and keep only those that are the last Friday of their month.

In [ ]:
def fetch_chain(currency):
    r = requests.get(f'{DERIBIT}/get_book_summary_by_currency',
                     params={'currency': currency, 'kind': 'option'}, timeout=20)
    r.raise_for_status()
    return r.json()['result']


def parse_name(name):
    parts = name.split('-')
    if len(parts) != 4:
        return None
    try:
        _, exp, strike, kind = parts
        dt = datetime.strptime(exp, '%d%b%y').replace(
            tzinfo=timezone.utc, hour=8, minute=0)
        return int(dt.timestamp() * 1000), float(strike), kind
    except (ValueError, IndexError):
        return None


raw = fetch_chain(CURRENCY)
now_ms = int(time.time() * 1000)

rows = []
for r in raw:
    p = parse_name(r['instrument_name'])
    if p is None: continue
    rows.append({
        'instrument': r['instrument_name'],
        'expiry': p[0],
        'strike': p[1],
        'kind': p[2],
        'days': (p[0] - now_ms) / 86_400_000,
        'mark': r.get('mark_price'),
        'bid': r.get('bid_price'),
        'ask': r.get('ask_price'),
        'iv': r.get('mark_iv'),
        'oi': r.get('open_interest') or 0,
        'volume': r.get('volume') or 0,
        'underlying': r.get('underlying_price'),
    })

df_all = pd.DataFrame(rows)
df_all['expiry_dt'] = pd.to_datetime(df_all.expiry, unit='ms', utc=True).dt.tz_convert(None)
df_all['iv_dec'] = df_all['iv'] / 100.0
df_all['spread_pct'] = (df_all['ask'] - df_all['bid']) / ((df_all['ask'] + df_all['bid']) / 2 + 1e-9)

spot = df_all['underlying'].dropna().iloc[0] if df_all['underlying'].notna().any() else df_all.strike.median()

# Find last Friday of each month in the future
def is_last_friday_of_month(dt):
    if dt.weekday() != 4:  # 4 = Friday
        return False
    next_friday = dt + timedelta(days=7)
    return next_friday.month != dt.month

all_expiries = df_all.groupby('expiry_dt').agg(
    contracts=('instrument', 'count'),
    days=('days', 'first'),
    total_oi=('oi', 'sum'),
    volume=('volume', 'sum'),
).reset_index().sort_values('days')
all_expiries = all_expiries[all_expiries.days > 0]

monthly_expiries = all_expiries[
    all_expiries.expiry_dt.apply(lambda d: is_last_friday_of_month(d.to_pydatetime()))
].reset_index(drop=True)

print(f'ETH spot: ${spot:,.2f}\n')
print('NEXT MONTHLY EXPIRIES (last Friday at 1:30 PM IST):\n')
for _, e in monthly_expiries.head(5).iterrows():
    ist = to_ist(e['expiry_dt'].to_pydatetime())
    print(f"  {ist.strftime('%A %d %B %Y - %I:%M %p IST')}  "
          f"({e['days']:.1f}d)  {e['contracts']} contracts  OI: {e['total_oi']:,.0f}")

## 2. Select the nearest monthly expiry

In [ ]:
if len(monthly_expiries) == 0:
    print('No future monthly expiries found')
else:
    target = monthly_expiries.iloc[0]
    monthly_dt = target['expiry_dt']
    ist_monthly = to_ist(monthly_dt.to_pydatetime())
    days_to_monthly = target['days']

    print(f'Target monthly expiry: {ist_monthly.strftime("%A %d %B %Y - %I:%M %p IST")}')
    print(f'Days away: {days_to_monthly:.1f}')
    print()

    df_monthly = df_all[df_all.expiry_dt == monthly_dt].copy()
    print(f'Contracts in this monthly: {len(df_monthly)}')
    print(f'Calls: {(df_monthly.kind=="C").sum()}  Puts: {(df_monthly.kind=="P").sum()}')

## 3. Monthly contract overview

In [ ]:
if len(monthly_expiries) > 0:
    print('=' * 65)
    print(f'MONTHLY EXPIRY: {ist_monthly.strftime("%A %d %B %Y - %I:%M %p IST")}')
    print('=' * 65)
    print(f'  Time remaining      : {days_to_monthly*24:.1f} hours ({days_to_monthly:.1f} days)')
    print(f'  Total contracts     : {len(df_monthly)}')
    print(f'  Unique strikes      : {df_monthly.strike.nunique()}')
    print(f'  Strike range        : ${df_monthly.strike.min():,.0f} - ${df_monthly.strike.max():,.0f}')
    print(f'  Call OI             : {df_monthly[df_monthly.kind=="C"].oi.sum():,.0f}')
    print(f'  Put OI              : {df_monthly[df_monthly.kind=="P"].oi.sum():,.0f}')
    print(f'  Total OI            : {df_monthly.oi.sum():,.0f}')
    print(f'  Volume today        : {df_monthly.volume.sum():,.0f}')
    pcr = df_monthly[df_monthly.kind=="P"].oi.sum() / max(df_monthly[df_monthly.kind=="C"].oi.sum(), 1)
    print(f'  PCR (OI)            : {pcr:.2f}')
    print(f'  ATM IV (nearest)    : {df_monthly.iloc[(df_monthly.strike - spot).abs().argsort()[:1]].iv_dec.iloc[0]*100:.1f}%')

## 4. Strike distribution

**What this tells you:** monthly OI is where institutional money sits. The strikes with the largest bars are the ones the market is defending.

In [ ]:
if len(monthly_expiries) > 0:
    near = df_monthly[(df_monthly.strike > spot * 0.7) & (df_monthly.strike < spot * 1.3)]
    calls = near[near.kind == 'C'].groupby('strike').oi.sum()
    puts  = near[near.kind == 'P'].groupby('strike').oi.sum()

    fig, ax = plt.subplots(figsize=(15, 5))
    all_strikes = sorted(set(calls.index) | set(puts.index))
    x = np.arange(len(all_strikes))
    c_vals = [calls.get(k, 0) for k in all_strikes]
    p_vals = [puts.get(k, 0) for k in all_strikes]

    ax.bar(x - 0.2, c_vals, 0.4, label='Call OI', color='seagreen')
    ax.bar(x + 0.2, p_vals, 0.4, label='Put OI', color='crimson')
    ax.axvline(np.interp(spot, all_strikes, x), color='black', lw=2, ls='--', label=f'spot ${spot:,.0f}')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{s:,.0f}' for s in all_strikes], rotation=90, fontsize=6)
    ax.set_ylabel('Open interest')
    ax.set_title(f'Monthly ETH {ist_monthly.strftime("%d %b")} - OI by strike')
    ax.legend()
    plt.tight_layout(); plt.show()

## 5. IV smile at monthly tenor

**Monthly IV is usually the lowest** of the three tenors (daily > weekly > monthly). It represents the market's **baseline volatility expectation** without short-term panic.

In [ ]:
if len(monthly_expiries) > 0:
    fig, ax = plt.subplots(figsize=(13, 5))
    c = df_monthly[(df_monthly.kind == 'C') & df_monthly.iv_dec.notna()].sort_values('strike')
    p = df_monthly[(df_monthly.kind == 'P') & df_monthly.iv_dec.notna()].sort_values('strike')
    if len(c) > 0:
        ax.plot(c.strike, c.iv_dec * 100, 'o-', color='seagreen', label='Call IV', lw=2, ms=4)
    if len(p) > 0:
        ax.plot(p.strike, p.iv_dec * 100, 'o-', color='crimson', label='Put IV', lw=2, ms=4)
    ax.axvline(spot, color='black', ls='--', lw=2, label=f'spot ${spot:,.0f}')
    ax.set_xlabel('strike'); ax.set_ylabel('IV %')
    ax.set_title(f'Monthly ETH IV smile - {ist_monthly.strftime("%d %b")}')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    daily_iv = df_all[df_all.days < 3].iv_dec.mean()
    weekly_iv = df_all[df_all.days.between(5, 9)].iv_dec.mean()
    monthly_iv = df_monthly.iv_dec.mean()
    print(f'\nIV comparison across tenors:')
    print(f'  Daily   (<3d)    : {daily_iv*100:.1f}%')
    print(f'  Weekly  (~7d)    : {weekly_iv*100:.1f}%')
    print(f'  Monthly (last Fri): {monthly_iv*100:.1f}%')

## 6. Greeks: delta, gamma, theta, vega

**Monthly options are the buyer's strategic tool:** low theta, high vega, moderate gamma. Perfect for longer thesis.

In [ ]:
def bs_greeks(S, K, T, sigma, kind):
    if T <= 0 or sigma <= 0:
        return None
    d1 = (np.log(S/K) + 0.5*sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    pdf = norm.pdf(d1)
    delta = norm.cdf(d1) if kind == 'C' else norm.cdf(d1) - 1
    gamma = pdf / (S * sigma * np.sqrt(T))
    vega  = S * pdf * np.sqrt(T) / 100
    theta = -(S * pdf * sigma) / (2 * np.sqrt(T)) / 365
    return {'delta': float(delta), 'gamma': float(gamma),
            'vega': float(vega), 'theta': float(theta)}

if len(monthly_expiries) > 0:
    greeks = []
    for _, r in df_monthly.iterrows():
        if pd.isna(r.iv_dec) or r.days <= 0: continue
        g = bs_greeks(spot, r.strike, r.days/365, r.iv_dec, r.kind)
        if g:
            greeks.append({**r.to_dict(), **g})
    gk = pd.DataFrame(greeks)
    print(f'Monthly contracts with Greeks: {len(gk)}')
    gk[['strike', 'kind', 'days', 'iv_dec', 'delta', 'gamma', 'vega', 'theta']].sort_values(['kind', 'strike']).head(15)

## 7. Theta decay - the slow bleed

**Monthly theta is the gentlest.** At 30 days out, an ATM option loses only 1-2% of premium per day. Compare that to daily options at 10-50%.

In [ ]:
if len(monthly_expiries) > 0:
    atm = gk.iloc[(gk.strike - spot).abs().argsort()[:1]].iloc[0]
    K, sigma = atm.strike, atm.iv_dec
    days_left = atm.days

    days_range = np.linspace(days_left, 0.1, 100)
    prices = []
    for d in days_range:
        T = d / 365
        if T <= 0: break
        d1 = (np.log(spot/K) + 0.5*sigma**2 * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        price = spot * norm.cdf(d1) - K * norm.cdf(d2)
        prices.append(price)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(days_range[:len(prices)], prices, lw=2, color='steelblue')
    ax.axhline(atm.mark, color='green', ls='--', lw=1, label=f'current ${atm.mark:.2f}')
    ax.axvline(0, color='red', lw=2, label='Monthly expiry (last Fri 1:30 PM IST)')
    ax.set_xlabel('days to expiry'); ax.set_ylabel('premium ($)')
    ax.set_title(f'ATM ${K:,.0f} call - monthly decay')
    ax.invert_xaxis(); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f'ATM ${K:,.0f} call:')
    print(f'  Current mark : ${atm.mark:.2f}')
    print(f'  Days left    : {days_left:.1f}')
    print(f'  Theta/day    : ${abs(atm.theta):.2f}')
    print(f'  Theta as % of premium: {abs(atm.theta)/atm.mark*100:.2f}% per day')
    print(f'  (Compare: daily ~10-50%, weekly ~2-5%)')

## 8. Breakeven per strike

**What this tells you:** ETH must reach this price by the **last Friday 1:30 PM IST** for the option to be profitable at expiry.

In [ ]:
if len(monthly_expiries) > 0:
    print('=' * 72)
    print('MONTHLY BREAKEVEN AT EXPIRY (last Friday 1:30 PM IST)')
    print('=' * 72)
    print()
    for kind_label, kind_code in [('CALLS', 'C'), ('PUTS', 'P')]:
        s = gk[(gk.kind == kind_code) & gk.mark.notna()].copy()
        s = s[(s.strike > spot * 0.90) & (s.strike < spot * 1.10)].sort_values('strike')
        if len(s) == 0: continue
        print(f'--- {kind_label} ---')
        for _, r in s.iterrows():
            be = r.strike + r.mark if kind_code == 'C' else r.strike - r.mark
            move = (be - spot) / spot * 100
            print(f'  ${r.strike:,.0f} {kind_code}  mark=${r.mark:.2f}  breakeven=${be:,.0f}  move: {move:+.2f}%')
        print()

## 9. Liquidity - the deepest of all tenors

**Monthly liquidity is 5-10x better than weekly.** Spreads are tight, OI is deep, you can size bigger.

In [ ]:
if len(monthly_expiries) > 0:
    liquid = df_monthly[(df_monthly.bid > 0) & (df_monthly.ask > 0)].copy()
    liquid['spread_%'] = liquid.spread_pct * 100
    near = liquid[(liquid.strike > spot * 0.95) & (liquid.strike < spot * 1.05)].sort_values('strike')

    print(f"{'Strike':<10} {'Kind':<5} {'Bid':<10} {'Ask':<10} {'Spread%':<10} {'OI':<10} {'Volume':<10}")
    print('-' * 70)
    for _, r in near.iterrows():
        print(f"${r.strike:,.0f}   {r.kind:<5} ${r.bid:<9.2f} ${r.ask:<9.2f} "
              f"{r['spread_%']:<10.1f} {r.oi:<10.0f} {r.volume:<10.0f}")

## 10. Best monthly contract for a buyer

**Scoring:** delta 0.30-0.50, high gamma/theta (efficiency), tight spread, deep OI.

In [ ]:
if len(monthly_expiries) > 0:
    candidates = liquid[(liquid.strike > spot * 0.90) &
                        (liquid.strike < spot * 1.10) &
                        (liquid.oi > 500)].copy()
    candidates = candidates.merge(gk[['strike', 'kind', 'delta', 'gamma', 'vega', 'theta']],
                                   on=['strike', 'kind'], how='left')
    candidates = candidates[candidates.delta.notna()]

    if len(candidates) > 0:
        candidates['delta_fit'] = 1 - abs(candidates.delta.abs() - 0.40) / 0.40
        candidates['gt_ratio'] = candidates.gamma / (candidates.theta.abs() + 1e-9)
        candidates['gt_norm'] = (candidates.gt_ratio - candidates.gt_ratio.min()) / (candidates.gt_ratio.max() - candidates.gt_ratio.min() + 1e-9)
        candidates['spread_score'] = 1 - candidates['spread_%'] / 100
        candidates['buyer_score'] = (
            0.40 * candidates.delta_fit +
            0.35 * candidates.gt_norm +
            0.25 * candidates.spread_score
        )

        best = candidates.nlargest(5, 'buyer_score')
        print('=' * 72)
        print('TOP 5 MONTHLY CONTRACTS FOR BUYERS')
        print('=' * 72)
        for _, r in best.iterrows():
            print(f"${r.strike:,.0f} {r.kind}  {r.days:.1f}d  score={r.buyer_score:.3f}")
            print(f"  mark=${r.mark:.2f}  spread={r['spread_%']:.1f}%  OI={r.oi:.0f}")
            print(f"  delta={r.delta:.3f}  gamma={r.gamma*10000:.2f}x10k  vega={r.vega:.2f}  theta=${r.theta:.2f}/day")
            print()

## 11. The monthly buyer's reality check

**Monthly is for strategic positions, not quick trades.** You have time, but you also have time for the market to move against you.

In [ ]:
if len(monthly_expiries) > 0:
    print('=' * 72)
    print('MONTHLY OPTION BUYER - REALITY CHECK')
    print('=' * 72)
    print()
    atm = gk.iloc[(gk.strike - spot).abs().argsort()[:1]].iloc[0]
    print(f'ATM ${atm.strike:,.0f} {atm.kind} option:')
    print(f'  Current mark       : ${atm.mark:.2f}')
    print(f'  Days to expiry     : {atm.days:.1f} (expires last Friday 1:30 PM IST)')
    print(f'  Theta/day          : ${abs(atm.theta):.2f} ({abs(atm.theta)/atm.mark*100:.2f}% of premium)')
    print(f'  Vega               : ${atm.vega:.2f} per 1% IV move')
    print()
    print(f'  Breakeven:')
    if atm.kind == 'C':
        be = atm.strike + atm.mark
    else:
        be = atm.strike - atm.mark
    move = (be - spot) / spot * 100
    print(f'    ETH must move to ${be:,.2f} ({move:+.2f}%) by monthly expiry')
    print()
    print('-' * 72)
    print('MONTHLY VS WEEKLY VS DAILY:')
    print('-' * 72)
    print()
    print('  DAILY:')
    print('    - Theta: brutal (10-50%/day), Gamma: highest, Liquidity: thin')
    print('    - For: 1-day catalysts only')
    print()
    print('  WEEKLY:')
    print('    - Theta: moderate (2-5%/day), Gamma: high, Liquidity: good')
    print('    - For: 3-7 day views')
    print()
    print('  MONTHLY:')
    print('    - Theta: lowest (1-2%/day), Gamma: moderate, Liquidity: deepest')
    print('    - For: 2-4 week strategic thesis')
    print()
    print('=' * 72)
    print('VERDICT: monthly is the STRATEGIC choice')
    print('=' * 72)
    print()
    print('  Deep liquidity + low theta + real vega.')
    print('  Use when you have a 2-4 week directional view.')
    print('  Also good when you expect IV to rise (high vega).')
    print('  Avoid if you need quick gamma or a 1-day trade.')
    print('  (For quick trades: buy weekly instead.)')

## 12. Summary

In [ ]:
if len(monthly_expiries) > 0:
    print('=' * 72)
    print('ETH MONTHLY OPTIONS - SUMMARY')
    print('=' * 72)
    print()
    print(f'  Spot                 : ${spot:,.2f}')
    print(f'  Monthly expiry       : {ist_monthly.strftime("%A %d %B - %I:%M %p IST")}')
    print(f'  Days away            : {days_to_monthly:.1f}')
    print(f'  Contracts            : {len(df_monthly)}')
    print(f'  Total OI             : {df_monthly.oi.sum():,.0f}')
    print(f'  PCR                  : {df_monthly[df_monthly.kind=="P"].oi.sum() / max(df_monthly[df_monthly.kind=="C"].oi.sum(), 1):.2f}')
    print(f'  ATM IV               : {df_monthly.iloc[(df_monthly.strike - spot).abs().argsort()[:1]].iv_dec.iloc[0]*100:.1f}%')
    print()
    print('=' * 72)
    print('BOTTOM LINE')
    print('=' * 72)
    print()
    print('  Monthly options expire on the last Friday of each month')
    print('  at 1:30 PM IST.')
    print()
    print('  They are the strategic choice:')
    print('  - Deepest liquidity')
    print('  - Lowest theta')
    print('  - Highest vega (sensitive to IV moves)')
    print('  - Moderate gamma (not explosive but real)')
    print()
    print('  When to buy monthly options:')
    print('  1. You have a 2-4 week directional thesis')
    print('  2. You expect IV to rise (buy low vega, sell high vega)')
    print('  3. You want to size up (liquidity supports it)')
    print('  4. The IV rank is low (cheap premium)')
    print()
    print('  When to skip monthly:')
    print('  - You need explosive gamma (buy weekly instead)')
    print('  - Your thesis is 1-3 days only')
    print('  - IV rank is very high (premium expensive)')
    print()
    print('=' * 72)

## Notes

**Monthly option characteristics:**

| Metric | Monthly | Weekly | Daily |
|---|---|---|---|
| Theta per day | 1-2% | 2-5% | 10-50% |
| Gamma | Moderate | High | Highest |
| Vega | Highest | Moderate | Very low |
| Liquidity | Deepest | Good | Thin |
| Spread | 1-4% | 3-8% | 5-15% |
| Expiry | Last Friday 1:30 PM IST | Every Friday 1:30 PM IST | Every day 1:30 PM IST |

**IST timing:**
- Monthly options settle on the **last Friday of each month at 1:30 PM IST** (08:00 UTC) 
- Quarterly months (March, June, September, December) have the largest monthly expiries
- The last Friday of December is often the largest expiry of the year

**Data notes:**
- Live from Deribit, no storage
- Expiries shown in IST throughout
- All Greeks computed with Black-Scholes